# Mean Sea Level Pressure — Northeast U.S.

This notebook downloads GFS MSLP, subsets to the Northeast U.S., and creates a cartopy map of mean sea level pressure.

Imports required libraries for data access, xarray handling, plotting, and cartopy mapping.

In [ ]:
import os
import requests
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

In [ ]:
url = "https://nomads.ncep.noaa.gov/cgi-bin/filter_gfs_1p00.pl?dir=%2Fgfs.20251103%2F18%2Fatmos&file=gfs.t18z.pgrb2.1p00.f006&var_MSLET=on&lev_mean_sea_level=on"
local_fname = "gfs.t18z.pgrb2.1p00.f006.mslet.grib2"
output_plot = "mslp_northeast.png"

lat_min, lat_max = 38, 46
lon_min, lon_max = -80, -66


### Function to download a file from the given URL if it does not already exist locally.

In [ ]:
def download_file(url, fname):
    if os.path.exists(fname):
        print(f"File already exists: {fname}")
        return
    print(f"Downloading {url} → {fname}")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(fname, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete.")

### Functions to open a GRIB2 file via cfgrib and to adjust longitudes from 0–360 to -180–180 when needed.

In [ ]:
def open_grib(fname):
    print(f"Opening GRIB2 file {fname}")
    ds = xr.open_dataset(fname, engine="cfgrib")
    print("Dataset variables:", list(ds.data_vars))
    return ds

def adjust_longitudes(ds):
    
    if 'longitude' not in ds.coords:
        return ds
    lons = ds.longitude
    try:
        if float(lons.max().values) > 180:
            ds = ds.assign_coords(longitude=((ds.longitude + 180) % 360) - 180)
            ds = ds.sortby('longitude')
    except Exception:
        
        pass
    return ds

### Subset routine to extract the region of interest and verify that the 'mslet' variable exists and contains data.

In [ ]:
def subset_region(ds, lat_min, lat_max, lon_min, lon_max):
    
    ds_sub = ds.sel(latitude=slice(lat_max, lat_min),
                    longitude=slice(lon_min, lon_max))
    if 'mslet' not in ds_sub:
        raise KeyError("mslet variable not found in dataset")
    if ds_sub['mslet'].size == 0:
        raise ValueError("Subset region returned zero points. Check your lat/lon bounds.")
    return ds_sub

### Plotting function to draw mean sea level pressure contours and filled contours using cartopy.

In [ ]:
def plot_mslp(ds, output_file=None):
    mslp = ds['mslet'] / 100.0  # Pa → hPa
    lons = ds['longitude']
    lats = ds['latitude']

    fig = plt.figure(figsize=(10,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.coastlines(resolution='50m')
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    try:
        ax.add_feature(cfeature.STATES, linestyle=':')
    except Exception:
        pass

    mslp_min = int(np.nanmin(mslp))
    mslp_max = int(np.nanmax(mslp)) + 2
    levels = list(range(mslp_min, mslp_max, 2))

    cs = ax.contour(lons, lats, mslp, levels=levels,
                    colors='black', linewidths=1, transform=ccrs.PlateCarree())
    cb = ax.contourf(lons, lats, mslp, levels=levels,
                     cmap='coolwarm', transform=ccrs.PlateCarree(), alpha=0.6)
    ax.clabel(cs, fmt='%d', inline=True, fontsize=8)
    plt.colorbar(cb, orientation='horizontal', pad=0.05, label='MSLP (hPa)')

    title_time = str(ds.time.values) if 'time' in ds else ""
    plt.title(f"Mean Sea Level Pressure – {title_time}")

    plt.tight_layout()
    if output_file:
        plt.savefig(output_file, dpi=150)
        print(f"Plot saved as {output_file}")
    plt.show()

### Utility to print a brief summary of the dataset and coordinate ranges.

In [ ]:
def print_dataset_info(ds):
    print('--- Dataset summary ---')
    print(ds)
    try:
        print('Latitude range:', float(ds.latitude.min().values), '→', float(ds.latitude.max().values))
        print('Longitude range:', float(ds.longitude.min().values), '→', float(ds.longitude.max().values))
    except Exception as e:
        print('Could not read coordinate ranges:', e)
    print('Variables:', list(ds.data_vars))

### Download step: will fetch the GRIB2 file if not already present.

In [ ]:
download_file(url, local_fname)

### Open the GRIB file, adjust longitudes to -180..180 if necessary, and display dataset info.

In [ ]:
ds = open_grib(local_fname)
ds = adjust_longitudes(ds)
print_dataset_info(ds)
ds

### Subset the opened dataset to the Northeast U.S. bounding box and show the shape of the result.

In [ ]:
ds_region = subset_region(ds, lat_min, lat_max, lon_min, lon_max)
print('Subset size:', ds_region['mslet'].shape)
ds_region

### Quick preview using basic matplotlib (no cartopy) to check the subset visually.

In [ ]:
arr = ds_region['mslet'] / 100.0
plt.figure(figsize=(6,4))
plt.contourf(ds_region.longitude, ds_region.latitude, arr, cmap='viridis')
plt.title('MSLP (hPa) — quick preview')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.colorbar(label='hPa')
plt.tight_layout()
plt.show()

### Generate and save the final cartopy map of MSLP for the subset region.

In [ ]:
plot_mslp(ds_region, output_plot)